# 04 — Gradient-based inverse design

Use the trained PHASE surrogate as a differentiable forward solver and optimize device geometry
to hit a target port behavior. The example here is a Y-branch with a 50/50 split target.

Optimizer: backprop through the Heun ODE sampler with gradient checkpointing
(see [`tools/inverse_design_ybranch_5050.py`](../tools/inverse_design_ybranch_5050.py) for the full,
production-grade version with extra losses, learning-rate schedules, and FDTD verification).

This notebook runs a short 50-iteration optimization for a quick demo.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
for p in (REPO_ROOT, REPO_ROOT / 'Model', REPO_ROOT / 'tools'):
    sp = str(p)
    if sp not in sys.path:
        sys.path.insert(0, sp)

CKPT_PATH = REPO_ROOT / 'checkpoints' / 'phase_300.pt'
print('ckpt:', CKPT_PATH, '(exists)' if CKPT_PATH.is_file() else '(MISSING)')

In [ ]:
import argparse
import inverse_design_ybranch_5050 as idy

parser = idy.build_argparser() if hasattr(idy, 'build_argparser') else None
if parser is None:
    print("This notebook talks to inverse_design_ybranch_5050.main directly.\n"
          "For a quick demo, run from the shell instead:\n"
          f"  python {idy.__file__} --ckpt {CKPT_PATH} --num-iters 50 --target 0.5")
else:
    args = parser.parse_args([
        '--ckpt', str(CKPT_PATH),
        '--num-iters', '50',
        '--target', '0.5',
        '--num-fm-steps', '20',
    ])
    print('args:', vars(args))

In [ ]:
import subprocess
# Easiest way to run is via the CLI:
cmd = [
    'python', str(REPO_ROOT / 'tools' / 'inverse_design_ybranch_5050.py'),
    '--ckpt', str(CKPT_PATH),
    '--num-iters', '50',
    '--target', '0.5',
    '--num-fm-steps', '20',
]
print(' '.join(cmd))
# subprocess.run(cmd, check=True)   # uncomment to launch (~5 min on A100)

## Inspect the result

The script writes optimization artifacts to `outputs/inverse_design_ybranch_5050/`. Below we
load the loss history and final geometry.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

out_dir = REPO_ROOT / 'outputs' / 'inverse_design_ybranch_5050'
history_path = out_dir / 'history.json'
final_path   = out_dir / 'final.npz'

if not history_path.is_file():
    print('No optimization run yet — uncomment the subprocess.run() above.')
else:
    history = json.loads(history_path.read_text())
    final   = np.load(final_path)
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2), constrained_layout=True)
    ax[0].plot(history['loss']); ax[0].set_yscale('log')
    ax[0].set_xlabel('iter'); ax[0].set_ylabel('loss'); ax[0].set_title('optimization')
    ax[1].imshow(final['eps_final'], origin='lower', cmap='viridis')
    ax[1].set_title('final geometry')
    mag = np.abs(final['ezr_final'] + 1j * final['ezi_final'])
    ax[2].imshow(mag, origin='lower', cmap='magma')
    ax[2].set_title(r'final $|E_z|$')
    plt.show()